<a href="https://colab.research.google.com/github/A-ghori/Pytorch-/blob/main/Revised_Neural_Net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [110]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [111]:
data = pd.read_csv("data.csv")
data.sample(15)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
271,8910748,B,11.290,13.04,72.23,388.0,0.09834,0.07608,0.03265,0.027550,...,16.18,78.27,457.5,0.1358,0.1507,0.12750,0.08750,0.2733,0.08022,NaN
213,881094802,M,17.420,25.56,114.50,948.0,0.10060,0.11460,0.16820,0.065970,...,28.07,120.40,1021.0,0.1243,0.1793,0.28030,0.10990,0.1603,0.06818,NaN
228,88147202,B,12.620,23.97,81.35,496.4,0.07903,0.07529,0.05438,0.020360,...,31.31,90.67,624.0,0.1227,0.3454,0.39110,0.11800,0.2826,0.09585,NaN
511,915664,B,14.810,14.70,94.66,680.7,0.08472,0.05016,0.03416,0.025410,...,17.58,101.70,760.2,0.1139,0.1011,0.11010,0.07955,0.2334,0.06142,NaN
447,9110944,B,14.800,17.66,95.88,674.8,0.09179,0.08890,0.04069,0.022600,...,22.74,105.90,829.5,0.1226,0.1881,0.20600,0.08308,0.3600,0.07285,NaN
449,911157302,M,21.100,20.52,138.10,1384.0,0.09684,0.11750,0.15720,0.115500,...,32.07,168.20,2022.0,0.1368,0.3101,0.43990,0.22800,0.2268,0.07425,NaN
11,84610002,M,15.780,17.89,103.60,781.0,0.09710,0.12920,0.09954,0.066060,...,27.28,136.50,1299.0,0.1396,0.5609,0.39650,0.18100,0.3792,0.10480,NaN
412,905539,B,9.397,21.68,59.75,268.8,0.07969,0.06053,0.03735,0.005128,...,27.99,66.61,301.0,0.1086,0.1887,0.18680,0.02564,0.2376,0.09206,NaN
486,913102,B,14.640,16.85,94.21,666.0,0.08641,0.06698,0.05192,0.027910,...,25.44,106.00,831.0,0.1142,0.2070,0.24370,0.07828,0.2455,0.06596,NaN
150,871001501,B,13.000,20.78,83.51,519.4,0.11350,0.07589,0.03136,0.026450,...,24.11,90.82,616.7,0.1297,0.1105,0.08112,0.06296,0.3196,0.06435,NaN


In [112]:
data.drop(columns=["id", "Unnamed: 32"], inplace=True)

In [113]:
X = data.drop(columns=["diagnosis"])
y = data["diagnosis"]

In [114]:
y

,diagnosis
0,M
1,M
2,M
3,M
4,M
...,...
564,M
565,M
566,M
567,M


In [115]:
X.shape
y.shape

(569,)

In [116]:
X_train,X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)


In [117]:
# Scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [118]:
# Label Encoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [119]:
# Numpy array to pytorch tensor

import torch

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)


In [120]:
X_train_tensor

tensor([[-1.4408, -0.4353, -1.3621,  ...,  0.9320,  2.0972,  1.8865],
        [ 1.9741,  1.7330,  2.0917,  ...,  2.6989,  1.8912,  2.4978],
        [-1.4000, -1.2496, -1.3452,  ..., -0.9702,  0.5976,  0.0579],
        ...,
        [ 0.0488, -0.5550, -0.0651,  ..., -1.2390, -0.7086, -1.2715],
        [-0.0390,  0.1021, -0.0314,  ...,  1.0500,  0.4343,  1.2134],
        [-0.5486,  0.3133, -0.6035,  ..., -0.6110, -0.3345, -0.8463]])

In [121]:
import torch
import torch.nn as nn

class Model(nn.Module):
  def __init__(self, num_features):

    super().__init__()

    self.loss_fn = torch.nn.BCELoss()
    self.Network = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128,1),
        nn.Sigmoid(),
    )
  def forward(self,features):
    out = self.Network(features)
    return out

  def loss_func(self,out, y):
    loss = self.loss_fn(out, y)
    return loss


In [122]:
learning_rate = 0.1
epochs = 1000

In [123]:
model = Model(X_train_tensor.shape[1])
for epoch in range(epochs):
  y_pred = model.forward(X_train_tensor)
  loss = model.loss_func(y_pred, y_train_tensor)

  loss.backward()
  with torch.no_grad():
    for param in model.parameters():
      param.data -= learning_rate * param.grad

     # Reset gradients
    # model.zero_grad()

    print(f"Epoch: {epoch+1}, Loss: {loss.item():.4f}")

  # with torch.no_grad():
  #   model.linear.weight -= learning_rate * model.linear.weight.grad
  #   model.linear.bias -= learning_rate * model.linear.weight.grad



Epoch: 1, Loss: 0.6890
Epoch: 2, Loss: 0.6178
Epoch: 3, Loss: 0.5058
Epoch: 4, Loss: 0.3908
Epoch: 5, Loss: 0.2961
Epoch: 6, Loss: 0.2286
Epoch: 7, Loss: 0.1846
Epoch: 8, Loss: 0.1571
Epoch: 9, Loss: 0.1398
Epoch: 10, Loss: 0.1278
Epoch: 11, Loss: 0.1180
Epoch: 12, Loss: 0.1085
Epoch: 13, Loss: 0.0990
Epoch: 14, Loss: 0.0900
Epoch: 15, Loss: 0.0825
Epoch: 16, Loss: 0.0774
Epoch: 17, Loss: 0.0756
Epoch: 18, Loss: 0.0777
Epoch: 19, Loss: 0.0833
Epoch: 20, Loss: 0.0915
Epoch: 21, Loss: 0.1004
Epoch: 22, Loss: 0.1084
Epoch: 23, Loss: 0.1138
Epoch: 24, Loss: 0.1154
Epoch: 25, Loss: 0.1129
Epoch: 26, Loss: 0.1072
Epoch: 27, Loss: 0.0992
Epoch: 28, Loss: 0.0901
Epoch: 29, Loss: 0.0817
Epoch: 30, Loss: 0.0756
Epoch: 31, Loss: 0.0723
Epoch: 32, Loss: 0.0705
Epoch: 33, Loss: 0.0697
Epoch: 34, Loss: 0.0689
Epoch: 35, Loss: 0.0682
Epoch: 36, Loss: 0.0670
Epoch: 37, Loss: 0.0648
Epoch: 38, Loss: 0.0608
Epoch: 39, Loss: 0.0544
Epoch: 40, Loss: 0.0456
Epoch: 41, Loss: 0.0363
Epoch: 42, Loss: 0.0315
E

In [124]:
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)

    predicted_labels = (y_pred >= 0.5).float()

    accuracy = (predicted_labels == y_test_tensor).float().mean()

print("Test Accuracy:", accuracy.item() * 100)

Test Accuracy: 95.61403393745422
